In [1]:
import pandas as pd
import json
from pathlib import Path
import numpy as np
from sklearn.preprocessing import normalize
import faiss

In [2]:
DATA_PATH = Path.cwd().parent / "data" / "rating_complete.csv"
MAPPING_PATH = Path.cwd().parent / "faiss_artifacts" / "mal_id_to_faiss_id.json"

In [3]:
ratings_df = pd.read_csv(DATA_PATH)
ratings_df.head()

,user_id,anime_id,rating
0,0,430,9
1,0,1004,5
2,0,3010,7
3,0,570,7
4,0,2762,9


In [4]:
len(ratings_df)

57633278

In [5]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57633278 entries, 0 to 57633277
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 1.3 GB


In [6]:
ratings_df.isnull().sum()

user_id     0
anime_id    0
rating      0
dtype: int64

In [7]:
ARTIFACT_DIR = Path.cwd().parent / "faiss_artifacts"
INDEX_PATH = ARTIFACT_DIR / "content_faiss_index.bin"

with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    mal_id_to_faiss_id = json.load(f)

mal_id_to_faiss_id = {int(k): int(v) for k, v in mal_id_to_faiss_id.items()}

In [8]:
index = faiss.read_index(str(INDEX_PATH))

In [9]:
ratings_filtered = ratings_df.dropna(subset=["anime_id", "rating"]).copy()
ratings_filtered["anime_id"] = ratings_filtered["anime_id"].astype(int)
ratings_filtered = ratings_filtered[ratings_filtered["anime_id"].isin(mal_id_to_faiss_id)]
ratings_filtered = ratings_filtered[ratings_filtered["rating"] > 0]

unique_anime_ids = ratings_filtered["anime_id"].unique()
faiss_ids = np.array([mal_id_to_faiss_id[anime_id] for anime_id in unique_anime_ids], dtype=np.int64)

# Reconstruct anime vectors from the content-based FAISS index
anime_vectors = np.vstack([index.reconstruct(int(i)) for i in faiss_ids]).astype("float32")
anime_id_to_vec = {anime_id: vec for anime_id, vec in zip(unique_anime_ids, anime_vectors)}

def compute_user_vector(group):
    vecs = np.stack([anime_id_to_vec[anime_id] for anime_id in group["anime_id"].to_numpy()])
    ratings = group["rating"].to_numpy(dtype="float32")
    weighted = (vecs * ratings[:, None]).sum(axis=0)
    return weighted / ratings.sum()

user_vectors = (
    ratings_filtered.groupby("user_id", sort=False)[["anime_id", "rating"]]
    .apply(compute_user_vector)
 )

user_ids = user_vectors.index.to_numpy()
user_matrix = np.vstack(user_vectors.to_list()).astype("float32")
user_matrix = normalize(user_matrix)

user_index = faiss.IndexHNSWFlat(user_matrix.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
user_index.hnsw.efConstruction = 200
user_index.add(user_matrix)

faiss.write_index(user_index, str(ARTIFACT_DIR / "user_faiss_index.bin"))

user_id_to_faiss_id = {int(user_id): int(i) for i, user_id in enumerate(user_ids)}
with open(ARTIFACT_DIR / "user_id_to_faiss_id.json", "w", encoding="utf-8") as f:
    json.dump(user_id_to_faiss_id, f)

user_id_to_vector = {int(user_id): user_matrix[i].tolist() for i, user_id in enumerate(user_ids)}
with open(ARTIFACT_DIR / "user_id_to_vector.json", "w", encoding="utf-8") as f:
    json.dump(user_id_to_vector, f)

len(user_id_to_faiss_id)

309484

In [10]:
def recommend_similar_users(user_id, n=5):
    user_id = int(user_id)

    if user_id not in user_id_to_faiss_id:
        print(f"User {user_id} not found.")
        return None

    query_idx = user_id_to_faiss_id[user_id]
    query_vec = user_matrix[query_idx:query_idx + 1]
    distances, indices = user_index.search(query_vec, n + 1)

    top_indices = [i for i in indices[0] if i != query_idx][:n]
    sim_scores = [d for i, d in zip(indices[0], distances[0]) if i != query_idx][:n]

    results = pd.DataFrame({
        "user_id": [int(user_ids[i]) for i in top_indices],
        "similarity": np.round(sim_scores, 3)
    })

    return results.reset_index(drop=True)

recommend_similar_users(user_ids[0], n=5)

,user_id,similarity
0,345897,0.933
1,125480,0.933
2,204567,0.933
3,223182,0.930
4,18670,0.930
